In [15]:
!pip install torch torchvision torchaudio
!pip install torch-geometric

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [16]:
import pandas as pd
import torch
import torch.nn.functional as F

from sklearn.preprocessing import LabelEncoder

from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

In [17]:
edges = pd.read_csv("datasets/elliptic_txs_edgelist.csv")

features = pd.read_csv(
    "datasets/elliptic_txs_features.csv",
    header=None
)

classes = pd.read_csv("datasets/elliptic_txs_classes.csv")

In [18]:
print(edges.shape)
print(features.shape)
print(classes.shape)

(234355, 2)
(203769, 167)
(203769, 2)


In [19]:
tx_ids = features.iloc[:,0].values

id_map = {
    tx_id: idx
    for idx, tx_id in enumerate(tx_ids)
}

print("Total Nodes:",len(id_map))

Total Nodes: 203769


In [20]:
edges = edges[
    edges["txId1"].isin(id_map) &
    edges["txId2"].isin(id_map)
]

edge_index = torch.tensor(
    [
        edges["txId1"].map(id_map).values,
        edges["txId2"].map(id_map).values
    ],
    dtype=torch.long
)

C:\Users\hanee\AppData\Local\Temp\ipykernel_20828\225433737.py:6: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\torch\csrc\utils\tensor_new.cpp:281.)
  edge_index = torch.tensor(


In [21]:
print(edge_index.shape)

print(edge_index.max())

print(edge_index.min())

torch.Size([2, 234355])
tensor(203768)
tensor(0)


In [22]:
x = torch.tensor(
    features.iloc[:,1:].values,
    dtype=torch.float
)

print(x.shape)

torch.Size([203769, 166])


In [23]:
encoder = LabelEncoder()

classes["class"] = encoder.fit_transform(
    classes["class"]
)

y = torch.tensor(
    classes["class"].values,
    dtype=torch.long
)

print(y.shape)

torch.Size([203769])


In [24]:
data = Data(
    x=x,
    edge_index=edge_index,
    y=y
)

print(data)

Data(x=[203769, 166], edge_index=[2, 234355], y=[203769])


In [25]:
class GraphSAGE(torch.nn.Module):

    def __init__(self):

        super().__init__()

        self.conv1 = SAGEConv(166,128)

        self.conv2 = SAGEConv(128,64)

        self.conv3 = SAGEConv(64,3)

    def forward(self,data):

        x=data.x

        edge_index=data.edge_index

        x=self.conv1(x,edge_index)

        x=F.relu(x)

        x=self.conv2(x,edge_index)

        x=F.relu(x)

        x=self.conv3(x,edge_index)

        return x

In [26]:
model=GraphSAGE()

optimizer=torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

loss_fn=torch.nn.CrossEntropyLoss()

In [27]:
for epoch in range(20):

    optimizer.zero_grad()

    out=model(data)

    loss=loss_fn(out,data.y)

    loss.backward()

    optimizer.step()

    print(
        f"Epoch {epoch+1}",
        "Loss:",
        loss.item()
    )

Epoch 1 Loss: 1.056574821472168
Epoch 2 Loss: 1.3110110759735107
Epoch 3 Loss: 0.7261874079704285
Epoch 4 Loss: 0.6750249862670898
Epoch 5 Loss: 0.6270949840545654
Epoch 6 Loss: 0.5326979756355286
Epoch 7 Loss: 0.5102037191390991
Epoch 8 Loss: 0.5119296908378601
Epoch 9 Loss: 0.5039469599723816
Epoch 10 Loss: 0.48238974809646606
Epoch 11 Loss: 0.46098124980926514
Epoch 12 Loss: 0.4498739242553711
Epoch 13 Loss: 0.4463600516319275
Epoch 14 Loss: 0.4415174424648285
Epoch 15 Loss: 0.43048685789108276
Epoch 16 Loss: 0.41826489567756653
Epoch 17 Loss: 0.41303497552871704
Epoch 18 Loss: 0.4111238420009613
Epoch 19 Loss: 0.40500107407569885
Epoch 20 Loss: 0.39548978209495544


In [28]:
pred=model(data)

prediction=pred.argmax(dim=1)

print(prediction[:20])

tensor([2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 2, 2, 1, 1, 2, 2, 1, 1, 2, 2])


In [30]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix

y_true = data.y.numpy()
y_pred = prediction.numpy()

print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, average="macro"))
print("Recall   :", recall_score(y_true, y_pred, average="macro"))
print("F1 Score :", f1_score(y_true, y_pred, average="macro"))

print("\nConfusion Matrix")

print(confusion_matrix(y_true, y_pred))

Accuracy : 0.8627367263911586
Precision: 0.5507296868316952
Recall   : 0.5245658143257519
F1 Score : 0.5345070920612601

Confusion Matrix
[[     0    225   4320]
 [     0  26117  15902]
 [     0   7523 149682]]


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [31]:
torch.save(
    model.state_dict(),
    "graphsage_model.pt"
)

print("GraphSAGE Model Saved Successfully!")

GraphSAGE Model Saved Successfully!
